# Push a CTranslate2 (faster-whisper) version of kanbior-fon-asr-v1

Downloads the latest model from `kanbior-fon-asr-v1` (transformers format), converts it to
**CTranslate2 float16**, checks it on a clip, and pushes it to a separate repo `kanbior-fon-asr-v1-ct2`.

float16 (~500 MB) works everywhere: pick the precision when loading.

```python
WhisperModel("<user>/kanbior-fon-asr-v1-ct2", device="cpu",  compute_type="int8")     # CPU
WhisperModel("<user>/kanbior-fon-asr-v1-ct2", device="cuda", compute_type="float16")  # GPU
```

Pushing needs a **write** token: `hf auth login` from the activated env (a read token can only download).
Needs ~2 GB free disk (1 GB download + 0.5 GB converted model).

In [1]:
SRC_REPO_NAME = "kanbior-fon-asr-v1"
SUBFOLDER = ""                 # "" = final model at the repo root; "last-checkpoint" during training
DST_REPO_NAME = "kanbior-fon-asr-v1-ct2"
QUANTIZATION = "float16"
PRIVATE = True

import json
import shutil
from pathlib import Path

from ctranslate2.converters import TransformersConverter
from faster_whisper import WhisperModel
from huggingface_hub import HfApi, snapshot_download

RUNS = Path("runs")
DATA_DIR = Path("sample-data")

api = HfApi()
user = api.whoami()["name"]
SRC_REPO, DST_REPO = f"{user}/{SRC_REPO_NAME}", f"{user}/{DST_REPO_NAME}"
print(f"{SRC_REPO}  ->  {DST_REPO}")

/Users/dinyad/Documents/IA/kanbior/.kanbior-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dyeto/kanbior-fon-asr-v1  ->  dyeto/kanbior-fon-asr-v1-ct2


## 1. Download the latest model

In [2]:
src_sha = api.model_info(SRC_REPO).sha    # exact source version, recorded in the commit and model card
download_dir = RUNS / "hub-download"       # overwritten on each download
prefix = f"{SUBFOLDER}/" if SUBFOLDER else ""
snapshot_download(
    SRC_REPO,
    revision=src_sha,
    local_dir=download_dir,
    allow_patterns=[f"{prefix}*"],
    # optimizer/scheduler state (~2 GB) is only needed to resume training
    ignore_patterns=["*optimizer*", "*scheduler*", "*rng_state*", "*scaler*", "*training_args*",
                     *([] if SUBFOLDER else ["last-checkpoint/*"])],
)
src = download_dir / SUBFOLDER if SUBFOLDER else download_dir
assert (src / "model.safetensors").is_file(), f"no model.safetensors in {src}: training not finished? try SUBFOLDER='last-checkpoint'"

state_file = src / "trainer_state.json"
step = json.loads(state_file.read_text())["global_step"] if state_file.is_file() else "final"
print(f"Source: {SRC_REPO}@{src_sha[:7]} / {SUBFOLDER or '(root)'}, step {step}")

Fetching 10 files: 100%|██████████| 10/10 [00:20<00:00,  2.05s/it]

Source: dyeto/kanbior-fon-asr-v1@4cf79b8 / (root), step final


## 2. Convert to CTranslate2

In [3]:
ct2_dir = RUNS / f"{DST_REPO_NAME}-{QUANTIZATION}"
copy = [f for f in ("tokenizer.json",) if (src / f).is_file()]
TransformersConverter(str(src), copy_files=copy).convert(str(ct2_dir), quantization=QUANTIZATION, force=True)
for f in sorted(ct2_dir.iterdir()):
    print(f"  {f.name:20s} {f.stat().st_size / 1e6:8.1f} MB")

W1123 12:42:05.668000 13784 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 479/479 [00:00<00:00, 6871.13it/s]


  config.json               0.0 MB
  model.bin               483.5 MB
  tokenizer.json            3.9 MB
  vocabulary.json           1.1 MB


## 3. Check it before pushing (CPU, loaded as int8)

In [4]:
import csv
import unicodedata

with open(DATA_DIR / "transcribed.csv", newline="", encoding="utf-8") as f:
    rows = [r for r in csv.DictReader(f) if (DATA_DIR / r["audio_path"]).is_file()][:2]

model = WhisperModel(str(ct2_dir), device="cpu", compute_type="int8")
for r in rows:
    segments, _ = model.transcribe(str(DATA_DIR / r["audio_path"]), language="fr", beam_size=1,
                                   without_timestamps=True)
    print(f"{r['id']}")
    print("  Reference:", unicodedata.normalize("NFC", r["transcription"]))
    print("  Predicted:", unicodedata.normalize("NFC", " ".join(s.text.strip() for s in segments)))
del model

[2026-11-23 12:42:13.846] [ctranslate2] [thread 152421] [warning] Tip: experimental MPS backend is available and often 2x+ faster than CPU on Apple Silicon (varies by workload and hardware). Try device="mps".


tr_000001
  Reference: "Kèmi" ná yi azɔ̌mɛ ɖo "Canada" é ná yí ɖo "semaine" e jǎwe ɔ́ mɛ.
  Predicted: Kɛ́mi ná yí azɔ̌mɛ ɖo Canada. É ná yí ɖo semaine e jǎwe ɔ́ mɛ.
tr_000002
  Reference: Un mɔ nyaví ɖokpó bɔ é ká nyí nyaví vɔvɔ ɖé lobɔ é ɖo sin ɖe dó nɛgbé wɛ.
  Predicted: Mǐ mɔ nyaví ɖokpó bɔ é ká nyaví vɔvɔ ɖé lobɔ é ɖo sin ɖé bló nɛgbé wɛ.


## 4. Model card and push

In [5]:
source_desc = f"{SUBFOLDER or 'final model'}" + (f", step {step}" if step != "final" else "")
card = f"""---
language:
- fon
license: cc-by-sa-4.0
library_name: ctranslate2
pipeline_tag: automatic-speech-recognition
base_model: {SRC_REPO}
tags:
- kanbior
- faster-whisper
- whisper
- fon
- speech-recognition
- kanbior
- ctranslate2
---

# {DST_REPO_NAME}

[CTranslate2](https://github.com/OpenNMT/CTranslate2) ({QUANTIZATION}) conversion of
[{SRC_REPO}](https://huggingface.co/{SRC_REPO}) (commit `{src_sha[:7]}`, {source_desc}),
a Fon speech recognition model fine-tuned from `openai/whisper-small`, for use with
[faster-whisper](https://github.com/SYSTRAN/faster-whisper).

## Usage

```python
from faster_whisper import WhisperModel

model = WhisperModel("{DST_REPO}", device="cpu", compute_type="int8")   # GPU: device="cuda", compute_type="float16"
segments, _ = model.transcribe("audio.wav", language="fr", beam_size=1, without_timestamps=True)
print(" ".join(s.text.strip() for s in segments))
```

Whisper has no Fon language token: the model was trained with the French token, so always pass
`language="fr"`. Output is Fon with tone marks.

## Training data

[JML-fon-speech-v1](https://huggingface.co/datasets/JaimeMaLangue/JML-fon-speech-v1) by J'aime Ma Langue
(CC-BY-SA-4.0, DOI 10.57967/hf/11698).

## License

[CC-BY-SA-4.0](https://creativecommons.org/licenses/by-sa/4.0/), the same license as the training data.
"""
(ct2_dir / "README.md").write_text(card, encoding="utf-8")

api.create_repo(DST_REPO, private=PRIVATE, exist_ok=True)
commit = api.upload_folder(
    repo_id=DST_REPO,
    folder_path=ct2_dir,
    commit_message=f"CT2 {QUANTIZATION} from {SRC_REPO}@{src_sha[:7]} ({source_desc})",
)
print("Pushed:", commit.commit_url)

Pushed: https://huggingface.co/dyeto/kanbior-fon-asr-v1-ct2/commit/272ea2b6ce20fa929ac437dcdc853bf1ba355f6f
